In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Dense,Input,Embedding,Conv1D,Bidirectional,LSTM,GRU,GlobalMaxPooling1D,Flatten
from tensorflow.keras.models import Model,Sequential
from tensorflow.keras.activations import relu,sigmoid
import keras_nlp

In [ ]:
import json
datastore=[]
with open('/kaggle/input/news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset.json','r') as f:
    for line in f:
        datastore.append(json.loads(line))

In [ ]:
len(datastore)

In [ ]:
datastore[13]

In [ ]:
sentences=[]
for item in datastore:
    sentences.append(item['headline'])

In [ ]:
len(sentences)

In [ ]:
labels=[]
for item in datastore:
    labels.append(item['is_sarcastic'])

In [ ]:
len(labels)

In [ ]:
TRAINING_SIZE=20000
train_sentences=sentences[0:TRAINING_SIZE]
train_labels=labels[0:TRAINING_SIZE]

test_sentences=sentences[TRAINING_SIZE:]
test_labels=labels[TRAINING_SIZE:]

In [ ]:
train_ds=tf.data.Dataset.from_tensor_slices(train_sentences)
keras_nlp.tokenizers.compute_word_piece_vocabulary(
    train_ds,
    vocabulary_size=10000,
    reserved_tokens=["[PAD]","[UNK]"],
    split='whitespace',
    vocabulary_output_file="sarcasm_vocab_subwords.txt"
)

In [ ]:
subword_tokenizer=keras_nlp.tokenizers.WordPieceTokenizer(
    vocabulary=('./sarcasm_vocab_subwords.txt')
)

In [ ]:
SHUFFLE_BUFFER_SIZE=10000
PREFETCH_BUFFER_SIZE=tf.data.AUTOTUNE
BATCH_SIZE=256
MAX_LENGTH=32
PADDING_TYPE='pre'
TRUNC_TYPE='post'


In [ ]:
def padded_func(sequences):
    sequences=sequences.ragged_batch(batch_size=sequences.cardinality())
    sequences=sequences.get_single_element()
    padded_sequences=tf.keras.utils.pad_sequences(
        sequences.numpy(),
        maxlen=MAX_LENGTH,
        truncating=TRUNC_TYPE,
        padding=PADDING_TYPE
        
    )
    padded_sequences=tf.data.Dataset.from_tensor_slices(padded_sequences)
    return padded_sequences

In [ ]:
train_dl=tf.data.Dataset.from_tensor_slices(train_labels)
test_ds=tf.data.Dataset.from_tensor_slices(test_sentences)
test_dl=tf.data.Dataset.from_tensor_slices(test_labels)

In [ ]:
train_sentences_subwords=train_ds.map(lambda text:subword_tokenizer.tokenize(text)).apply(padded_func)
test_sentences_subwords=test_ds.map(lambda text:subword_tokenizer.tokenize(text)).apply(padded_func)

In [ ]:
train_dataset_vectorized=tf.data.Dataset.zip(train_sentences_subwords,train_dl)
test_dataset_vectorized=tf.data.Dataset.zip(test_sentences_subwords,test_dl)

In [ ]:
train_dataset_final=(train_dataset_vectorized
                    .cache()
                    .shuffle(SHUFFLE_BUFFER_SIZE)
                    .prefetch(PREFETCH_BUFFER_SIZE)
                    .batch(BATCH_SIZE))

test_dataset_final=(test_dataset_vectorized
                   .cache()
                   .shuffle(SHUFFLE_BUFFER_SIZE)
                   .batch(BATCH_SIZE))

In [ ]:
subword_tokenizer.vocabulary_size()

In [ ]:
def plot_loss_acc(history):
  '''Plots the training and validation loss and accuracy from a history object'''
  acc = history.history['accuracy']
  val_acc = history.history['val_accuracy']
  loss = history.history['loss']
  val_loss = history.history['val_loss']

  epochs = range(len(acc))

  fig, ax = plt.subplots(1,2, figsize=(12, 6))
  ax[0].plot(epochs, acc, 'bo', label='Training accuracy')
  ax[0].plot(epochs, val_acc, 'b', label='Validation accuracy')
  ax[0].set_title('Training and validation accuracy')
  ax[0].set_xlabel('epochs')
  ax[0].set_ylabel('accuracy')
  ax[0].legend()

  ax[1].plot(epochs, loss, 'bo', label='Training Loss')
  ax[1].plot(epochs, val_loss, 'b', label='Validation Loss')
  ax[1].set_title('Training and validation loss')
  ax[1].set_xlabel('epochs')
  ax[1].set_ylabel('loss')
  ax[1].legend()

  plt.show()

In [ ]:
# Parameters
EMBEDDING_DIM = 16
LSTM_DIM = 32
DENSE_DIM = 24
VOCAB_SIZE=10000

# Model Definition with LSTM
model_lstm = tf.keras.Sequential([
    tf.keras.Input(shape=(MAX_LENGTH,)),
    tf.keras.layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(LSTM_DIM)),
    tf.keras.layers.Dense(DENSE_DIM, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Set the training parameters
model_lstm.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

# Print the model summary
model_lstm.summary()

In [ ]:
NUM_EPOCHS = 10

# Train the model
history_lstm = model_lstm.fit(train_dataset_final, epochs=NUM_EPOCHS, validation_data=test_dataset_final)

In [ ]:
plot_loss_acc(history_lstm)

In [ ]:
# Parameters
EMBEDDING_DIM = 16
LSTM_DIM1 = 128
LSTM_DIM2=64
DENSE_DIM = 24
VOCAB_SIZE=10000

# Model Definition with LSTM
model_two_lstm = tf.keras.Sequential([
    tf.keras.Input(shape=(MAX_LENGTH,)),
    tf.keras.layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(LSTM_DIM1,return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(LSTM_DIM2)),
    tf.keras.layers.Dense(DENSE_DIM, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Set the training parameters
model_two_lstm.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

# Print the model summary
model_two_lstm.summary()

In [ ]:
NUM_EPOCHS = 10

# Train the model
history_lstm = model_two_lstm.fit(train_dataset_final, epochs=NUM_EPOCHS, validation_data=test_dataset_final)

In [ ]:
plot_loss_acc(history_lstm)

In [ ]:
FILTERS=128
KERNEL_SIZE=5
model_conv=Sequential([
    Input(shape=(MAX_LENGTH,)),
    Embedding(input_dim=VOCAB_SIZE,output_dim=EMBEDDING_DIM),
    Conv1D(FILTERS,KERNEL_SIZE,activation='relu'),
    GlobalMaxPooling1D(),
    Dense(6,activation='relu'),
    Dense(1,activation='sigmoid')
])
model_conv.summary()

In [ ]:
model_conv.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [ ]:
history_conv=model_conv.fit(train_dataset_final,epochs=NUM_EPOCHS,validation_data=test_dataset_final)

In [ ]:
plot_loss_acc(history_conv)